# NACE 2 industry classification table (`TLS229_APPLN_NACE2`)

Welcome to this look at the NACE 2 industry classification table in the PATSTAT database: the **NACE 2 industry classification table**, designated as `TLS229_APPLN_NACE2`. This table indicates the extent to which an application is associated with one or more industries.
It is generated using the reference table `TLS902_IPC_NACE2` and the IPCs assigned to each application. As a result, NACE2 codes are not assigned to applications without IPCs.

Note: in the reference table `TLS902_IPC_NACE2` the IPC codes are mapped to NACE codes, which represent only manufacturing industries. Additionally, the table `TLS229_APPLN_NACE2` contains all applications, including  the ones that are filed by applicants that are clearly not manufacturers, such as hospitals, universities or organisations. For this reason, depending on the type of analysis one intends to carry out, the creation of a custom mapping to NACE codes may be needed.

Let’s break down the key fields in this table. The first step is to initialise the PATSTAT client and import the table.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the models
from epo.tipdata.patstat.database.models import TLS229_APPLN_NACE2

## Key fields in the `TLS229_APPLN_NACE2` table

* `APPLN_ID` (primary key): it is the unique identifier for each patent application in the PATSTAT database.
* `NACE2_CODE`: NACE2 is the Statistical Classification of Economics Activities in the European Community. `NACE2_CODE`is the code associated with a defined category of economic activity.

To identify, for instance, the ``NACE2_CODE`` for each application kind (`APPLN_KIND`), we could join the `TLS229_APPLN_NACE2`with the `TLS201_APPLN`.

In [4]:
#Import the two tables TLS201_APPL and TLS229_APPLN_NACE2 
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS229_APPLN_NACE2 

#Join the two tables using the appln_id in order to retrieve the NACE code for each application kind.
nace_id = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS229_APPLN_NACE2.appln_id,
        TLS229_APPLN_NACE2.nace2_code,
    )
    .join(
        TLS201_APPLN,
        TLS201_APPLN.appln_id==TLS229_APPLN_NACE2.appln_id,
    )
    .limit(10000)
)

#Show the results as a dataframe
nace_kind_df= patstat.df(nace_id)
nace_kind_df

,appln_id,appln_auth,appln_id,nace2_code
0,22480945,GB,22480945,28.9
1,2993304,BE,2993304,22
2,633357020,CN,633357020,30
3,53681754,US,53681754,28.9
4,27784718,JP,27784718,18.1
...,...,...,...,...
9995,555854914,CN,555854914,26.2
9996,25368772,JP,25368772,26.51
9997,612628631,CN,612628631,25.9
9998,489773240,CN,489773240,20.1


* `WEIGHT`: another key field in the ``TLS229_APPLN_NACE2`` table is the `weight`. The weight measures the extent to which each application belongs to one or more industries. For each application ID, the weight can assume values between 0 and 1. The sum of all weights for a given application always equals 1. The closer the value is to 1, the more the application belongs to the industry identified by the corresponding NACE2 code.
For instance, if an application has a weight equal to 1 for the NACE2 code 23.3 (Manufacture of clay building materials), this means that the application exclusively and entirely belongs to that industry. On the other hand, if an application has a weight of 0.5 for the NACE2 code 23.3 (Manufacture of clay building materials) and a weight of 0.5 for another NACE2 code, for example 22.2 (Manufacture of plastic products), this means that the application belongs to the same extent to both the 22.2 and the 23.3 industries.

To illustrate this, we can retrieve the weights associated with a randomly selected application in order to observe the extent to which it belongs to one or more industries.
First, to obtain a clearer overview, we join the table ``TLS229_APPLN_NACE2`` with the table ``TLS902_IPC_NACE2`` in order to retrieve the descriptions of the NACE2 codes associated with each application.

In [5]:
#Import the two tables TLS229_APPLN_NACE2 and TLS902_IPC_NACE2
from epo.tipdata.patstat.database.models import TLS229_APPLN_NACE2, TLS902_IPC_NACE2
#Join the two tables using the nace2_code in order to retrieve the nace2 description (nace2_descr) for each application kind.
nace_descr_appid = (
    db.query(
        TLS229_APPLN_NACE2.appln_id,
        TLS229_APPLN_NACE2.nace2_code,
        TLS229_APPLN_NACE2.weight,
        TLS902_IPC_NACE2.nace2_descr,
    )
    .join(
        TLS902_IPC_NACE2,
        TLS902_IPC_NACE2.nace2_code==TLS229_APPLN_NACE2.nace2_code,
    )
    .order_by(
        TLS229_APPLN_NACE2.appln_id
    )
    .distinct()  #Add " distinct" to remove duplicate
    .limit(10000)
)
     
#Show the results as a dataframe
nace_descr_appid_df= patstat.df(nace_descr_appid)
nace_descr_appid_df

,appln_id,nace2_code,weight,nace2_descr
0,1,27.33,0.111111,Manufacture of Wiring Devices
1,1,26.3,0.555556,Manufacture of Communication Equipment
2,1,28.23,0.333333,Manufacture of Office Machinery and Equipment ...
3,2,26.5,0.210526,Manufacture of Instruments and Appliances for ...
4,2,21,0.789474,Manufacture of Basic Pharmaceutical Products a...
...,...,...,...,...
9995,9511,26.5,1.000000,Manufacture of Instruments and Appliances for ...
9996,9513,28.9,1.000000,Manufacture of Other Special-Purpose Machinery
9997,9515,32.5,1.000000,Manufacture of medical and dental instruments ...
9998,9517,26.2,0.500000,Manufacture of computers and peripheral equipment


Now let's retrieve a single application from the dataframe to see to which industries it belongs and to what extent.

In [5]:
from epo.tipdata.patstat.database.models import TLS229_APPLN_NACE2, TLS902_IPC_NACE2

# Let's recall the application 380098085 to see to which industries belongs
app_id = 380098085 

nace_per_app = (
    db.query(
        TLS229_APPLN_NACE2.appln_id,
        TLS229_APPLN_NACE2.nace2_code,
        TLS902_IPC_NACE2.nace2_descr,
        TLS229_APPLN_NACE2.weight
    )
    .join(
        TLS902_IPC_NACE2,
        TLS902_IPC_NACE2.nace2_code == TLS229_APPLN_NACE2.nace2_code
    )
    .filter(
        TLS229_APPLN_NACE2.appln_id == app_id   
    )  
    .distinct()  
    .limit(10000)

    
)

# Show the result as a dataframe
nace_per_app_df = patstat.df(nace_per_app)
nace_per_app_df

,appln_id,nace2_code,nace2_descr,weight
0,380098085,27.5,Manufacture of Domestic Appliances,1.0


As can be seen from the output, application 380098085 belongs to only one industry, Manufacture of domestic appliances, corresponding to the NACE2 code 27.5.  As expected, the weight in this case is equal to 1.

Let us now examine how the output changes when we retrieve an application belonging to multiple industries, for example application 4539356.

In [8]:
from epo.tipdata.patstat.database.models import TLS229_APPLN_NACE2, TLS902_IPC_NACE2

# Let's recall application 4539356 to see which industries it belongs to
app_id = 4539356 

nace_per_app = (
    db.query(
        TLS229_APPLN_NACE2.appln_id,
        TLS229_APPLN_NACE2.nace2_code,
        TLS902_IPC_NACE2.nace2_descr,
        TLS229_APPLN_NACE2.weight
    )
    .join(
        TLS902_IPC_NACE2,
        TLS902_IPC_NACE2.nace2_code == TLS229_APPLN_NACE2.nace2_code
    )
    .filter(
        TLS229_APPLN_NACE2.appln_id == app_id   
    )  # Filters for a single application (in this example, 380098085)

    .distinct()  
    .limit(10000)

    
)

# Show the result as a dataframe
nace_per_app_df = patstat.df(nace_per_app)
nace_per_app_df

,appln_id,nace2_code,nace2_descr,weight
0,4539356,43,Specialised Construction Activities,0.571429
1,4539356,25.9,Manufacture of Other Fabricated Metal Products,0.428571


As we can see from the output, application 4539356 belongs to both the Manufacture of other metal products (NACE2 code 25.9) and the Specialised construction activities (NACE2 code 43) industries.

Notice that the weight associated with Specialised construction activities is greater than the weight associated with Manufacture of other metal products (0.571429 > 0.428571), indicating that the application belongs to the Specialised construction activities industry to a greater extent.

Let us plot both applications to visualize the results more clearly.

In [7]:
from epo.tipdata.patstat.database.models import TLS229_APPLN_NACE2, TLS902_IPC_NACE2

# Let's recall the applications 4539356 and 380098085 to see to which industries belongs
app_ids =[ 4539356, 380098085] 

nace_per_app = (
    db.query(
        TLS229_APPLN_NACE2.appln_id,
        TLS229_APPLN_NACE2.nace2_code,
        TLS902_IPC_NACE2.nace2_descr,
        TLS229_APPLN_NACE2.weight
    )
    .join(
        TLS902_IPC_NACE2,
        TLS902_IPC_NACE2.nace2_code == TLS229_APPLN_NACE2.nace2_code
    )
    .filter(
        TLS229_APPLN_NACE2.appln_id.in_(app_ids)
    ) 
    .order_by(
        TLS229_APPLN_NACE2.nace2_code
    )
    .distinct()  
    .limit(10000)

    
)

# Show the result as a dataframe
nace_per_app_df = patstat.df(nace_per_app)
nace_per_app_df

,appln_id,nace2_code,nace2_descr,weight
0,4539356,25.9,Manufacture of Other Fabricated Metal Products,0.428571
1,380098085,27.5,Manufacture of Domestic Appliances,1.000000
2,4539356,43,Specialised Construction Activities,0.571429
